Qwen3 From Scratch (A Standalone Notebook)

In [ ]:
# 检查关键依赖库是否已安装及其版本号,确保后续代码能在这些版本下正常运行
from importlib.metadata import version

# 逐一列出本 notebook 依赖的第三方库并打印版本
pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# Select which model to use via the following flag; only one can be True

# 三选一: 基础模型(未微调)/ 推理模型(reasoning,支持 <think> 思考过程)/ 指令微调模型(instruct)
USE_BASE_MODEL = False
USE_REASONING_MODEL = True
USE_INSTRUCT_MODEL = False

# 利用布尔值可当整数相加的特性,校验三个开关中有且仅有一个为 True
if (USE_BASE_MODEL + USE_REASONING_MODEL
    + USE_INSTRUCT_MODEL) != 1:
    raise AttributeError("Only one of the options above can be True.")

1. Architecture code

In [ ]:
# ===== 模型核心组件: FeedForward(SwiGLU)、RMSNorm、RoPE(旋转位置编码)、
# 分组查询注意力 GroupedQueryAttention(含 QK-Norm)、TransformerBlock、Qwen3Model 以及 KVCache =====
import torch
import torch.nn as nn


# SwiGLU 前馈网络: fc1(门控分支)过 SiLU 激活后,与 fc2(值分支)逐元素相乘,再经 fc3 投影回 emb_dim,
# 表达能力优于传统的 Linear+ReLU+Linear,是目前主流 LLM(LLaMA/Qwen 等)的标准设计
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)
        # fc1、fc2 把 emb_dim 投影到 hidden_dim,分别作为 SwiGLU 的“门控”和“值”分支(均无偏置);fc3 把 hidden_dim 投影回 emb_dim

    def forward(self, x):
        # x: (batch_size, num_tokens, emb_dim)
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        # SwiGLU: SiLU(fc1(x)) 逐元素乘以 fc2(x),得到形状 (batch_size, num_tokens, hidden_dim)
        return self.fc3(x)
# RMSNorm(均方根归一化): 相比 LayerNorm 去掉了减均值这一步,只用均方根(RMS)做缩放,计算更省、效果相当,
# 是当前主流大模型广泛采用的归一化方式
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        # x 的最后一维是待归一化的维度(可以是 emb_dim,也可以是 GQA 中单个头的 head_dim,取决于调用处)
        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)
        # 提升到 float32 计算方差,避免 bfloat16/float16 精度不足带来的数值误差,这是与官方 Qwen3 实现对齐所需的细节

        # RMS = sqrt(mean(x^2)),不像 LayerNorm 那样先减去均值;keepdim=True 便于结果广播回原始形状
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale
        # self.scale 是可学习的逐通道缩放参数,对应 HF 权重里的 *.weight(如 q_norm.weight / k_norm.weight / input_layernorm.weight 等)

        if self.shift is not None:
            norm_x = norm_x + self.shift

        return norm_x.to(input_dtype)
# 预计算 RoPE(旋转位置编码)所需的 cos/sin 表。RoPE 通过对 Q、K 向量按位置角度做旋转来注入位置信息,
# 不需要额外的可学习参数,且天然支持长度外推
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))
    # inv_freq 形状为 (head_dim // 2,),每一维对应一个不同的旋转频率(theta_base 越大,低频分量衰减越慢,越有利于长上下文)

    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)
    # 位置索引与逆频率做外积,得到每个位置、每个频率对应的旋转角度

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)
    # 把角度矩阵在最后一维复制一份拼接,使其维度与 head_dim 对齐,方便后面直接对整个头向量做旋转(前后两半共用同一组角度)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


# 对 Q 或 K 向量应用 RoPE 旋转。核心是“旋转一半”(rotate half)技巧: 把向量切成前后两半 (x1, x2),
# 用 (x1,x2) -> (x1*cos - x2*sin, x2*cos + x1*sin) 的方式做二维旋转,等价于对每一对维度施加不同角度的复数旋转
def apply_rope(x, cos, sin, offset=0):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : head_dim // 2]  # First half
    # x1、x2 分别是 head_dim 维向量的前半段和后半段,形状均为 (batch_size, num_heads, seq_len, head_dim // 2)
    x2 = x[..., head_dim // 2:]  # Second half

    # Adjust sin and cos shapes
    # offset 是这段序列在整个上下文中的起始绝对位置。使用 KV cache 增量解码时,新 token 并非从位置 0 开始,
    # 而是从 start_pos 开始,因此必须用 offset 从预计算好的 cos/sin 表中取出对应位置的角度,才能得到正确的位置编码
    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    # 构造“旋转后”的向量 (-x2, x1),再与原始 x 分别乘以 sin、cos 后相加,即完成旋转位置编码的注入
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)
# 分组查询注意力(GQA): 把 num_heads 个 Query 头划分成 num_kv_groups 组,组内的多个 Query 头共享同一份 Key/Value,
# 从而大幅减少 K、V 的显存占用(尤其是自回归生成时的 KV cache),效果上介于
# 完整多头注意力(MHA, num_kv_groups == num_heads)与多查询注意力(MQA, num_kv_groups == 1)之间
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups
        # group_size: 每一组 K/V 被多少个 Query 头共享,后面会用 repeat_interleave 把 K/V 复制 group_size 份来对齐 Query 头数

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        # 注意 W_query 的输出维度是 num_heads * head_dim,而 W_key/W_value 的输出维度只有 num_kv_groups * head_dim,
        # 这正是 GQA 相比标准多头注意力节省参数量与显存的地方

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        # QK-Norm: Qwen3 相比早期模型新增的技巧,对每个头的 Q、K 向量(维度为 head_dim)分别做一次 RMSNorm 再参与注意力计算,
        # 有助于稳定训练、抑制注意力分数出现极端值
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape
        # x: (b, num_tokens, d_in); start_pos: 当前这批 token 在整个序列中的起始绝对位置(用于 RoPE 和 KV cache 对齐);
        # cache: 该层此前缓存的 (key, value),形状均为 (b, num_kv_groups, 已缓存的seq_len, head_dim),首次调用为 None

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)
        # 这里只对当前输入的 num_tokens 个新 token 计算 Q/K/V 投影;若使用 KV cache,解码阶段 num_tokens 通常为 1

        # Reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys_new = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values_new = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        # 拆分成多头并转置: queries -> (b, num_heads, num_tokens, head_dim);keys_new/values_new -> (b, num_kv_groups, num_tokens, head_dim)
        # 注意 Query 的头数与 Key/Value 的“组数”在这一步是不同的,这正是 GQA 的形状体现

        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys_new = self.k_norm(keys_new)
        # QK-Norm 是在“拆分成每个头”之后、在维度为 head_dim 的最后一维上做归一化,顺序是: 先做 QK-Norm,再做 RoPE

        # Apply RoPE
        queries = apply_rope(queries, cos, sin, offset=start_pos)
        keys_new = apply_rope(keys_new, cos, sin, offset=start_pos)
        # 用 start_pos 作为 offset,保证增量解码时新 token 使用的是它在完整序列中的真实绝对位置所对应的旋转角度

        if cache is not None:
            prev_k, prev_v = cache
            keys = torch.cat([prev_k, keys_new], dim=2)
            values = torch.cat([prev_v, values_new], dim=2)
            next_cache = (keys, values)
        # KV Cache 的核心: 把历史缓存的 prev_k/prev_v 与本次新算出的 keys_new/values_new 沿序列长度维(dim=2)拼接,
        # 得到到目前为止全部位置的完整 K、V,而不需要重新计算历史 token 的 K、V,这是增量解码能提速的关键
        else:
            start_pos = 0  # reset RoPE
            keys, values = keys_new, values_new
            next_cache = (keys, values)
        # 备注: 上面这行对 start_pos 的重置是无实际作用的历史遗留代码——RoPE 已在此前用原始 start_pos 计算完毕,
        # 这里的重新赋值之后不会再被使用,不影响结果正确性

        # Expand K and V to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)
        # 把 num_kv_groups 个头沿头维度(dim=1)重复 group_size 次,扩展成 num_heads 份,
        # 使每个 Query 头都能找到与之对应(同组共享)的 Key/Value,形状变为 (b, num_heads, 完整seq_len, head_dim)

        # Attention
        # 缩放点积注意力: (b, num_heads, num_tokens, head_dim) @ (b, num_heads, head_dim, 完整seq_len)
        # -> attn_scores 形状为 (b, num_heads, num_tokens, 完整seq_len)
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        # 除以 sqrt(head_dim) 做缩放,避免点积结果过大导致 softmax 梯度消失

        # 加权求和后把多头拼回一起: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads*head_dim)
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context), next_cache
# 一个标准的 Pre-Norm Transformer 块: 先归一化再送入子层,子层输出与归一化前的输入做残差相加(shortcut)
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        # Shortcut connection for attention block
        # 把 start_pos、cache 原样透传给注意力子层,并接收该层更新后的 next_cache 向外返回,供上层 KVCache 容器保存
        shortcut = x
        x = self.norm1(x)
        x, next_cache = self.att(x, mask, cos, sin, start_pos=start_pos, cache=cache)  # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # Add the original input back

        return x, next_cache
# 完整的 Qwen3 模型: 词嵌入 + N 层 TransformerBlock + 最终归一化 + 输出头,并内置了对 KV cache 增量解码的支持
class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Main model parameters
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.trf_blocks = nn.ModuleList(  # ModuleList since Sequential can only accept one input, and we need `x, mask, cos, sin`
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # Reusable utilities
        if cfg["head_dim"] is None:
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        # cos/sin 表在整个模型中只计算一次,所有层、所有头共享同一份位置编码表;
        # persistent=False 表示它们不会被写入 state_dict/checkpoint(可以随时按 head_dim/context_length 重新计算出来)
        self.cfg = cfg
        self.current_pos = 0  # Track current position in KV cache
        # current_pos 记录“到目前为止已经喂给模型多少个 token”,每次带 cache 的 forward 调用后会自动累加,
        # 从而在下一次调用时知道新 token 的起始绝对位置(start_pos)

    def forward(self, in_idx, cache=None):
        # cache=None: 普通/训练模式的整段前向(一次性处理完整序列,使用标准的下三角因果掩码)
        # cache 不为 None: 增量解码模式,in_idx 通常只包含新 token,依赖 KVCache 保存的历史 K/V
        # Forward pass
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        if cache is not None:
            pos_start = self.current_pos
            pos_end = pos_start + num_tokens
            self.current_pos = pos_end
            mask = torch.triu(
                torch.ones(pos_end, pos_end, device=x.device, dtype=torch.bool), diagonal=1
            )[pos_start:pos_end, :pos_end]
        # 增量解码下的因果掩码: 先构造完整 pos_end x pos_end 的上三角掩码,再只切出新 token(第 pos_start 到 pos_end 行)
        # 对全部 0..pos_end-1 个位置的可见性,保证新 token 能看到所有历史 token,同时维持因果(不看未来)约束
        else:
            pos_start = 0  # Not strictly necessary but helps torch.compile
            mask = torch.triu(
                torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1
            )
        # 不使用 cache 时就是标准的一次性因果掩码: 每个位置只能看到它自己及之前的位置
        # Shape (1, 1, num_tokens, num_tokens) to broadcast across batch and heads
        mask = mask[None, None, :, :]

        for i, block in enumerate(self.trf_blocks):
            blk_cache = cache.get(i) if cache else None
            x, new_blk_cache = block(x, mask, self.cos, self.sin,
                                     start_pos=pos_start,
                                     cache=blk_cache)
            if cache is not None:
                cache.update(i, new_blk_cache)
        # 逐层取出该层此前的 KV 缓存传入,再用该层返回的新缓存更新 KVCache 容器,各层的缓存彼此独立保存

        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

    def reset_kv_cache(self):
        self.current_pos = 0
        # 开始生成一个新的序列前需要调用此方法,把位置计数器清零(注意它并不会清空 KVCache 对象里已保存的张量,
        # 实际使用时通常会像下面 generate_text_basic_stream 那样重新创建一个新的 KVCache 实例)
# KV Cache 容器: 为每一层保存一份 (key, value) 元组,在自回归生成过程中逐步增长,避免每一步都重新计算全部历史的 K/V
class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None

2. Initialize model

In [ ]:
# 选择要加载的 Qwen3 模型规模,不同规模对应不同的超参数配置
CHOOSE_MODEL = "0.6B"

if CHOOSE_MODEL == "0.6B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,           # Vocabulary size
        "context_length": 40_960,        # Context length that was used to train the model
        "emb_dim": 1024,                 # Embedding dimension
        "n_heads": 16,                   # Number of attention heads
        "n_layers": 28,                  # Number of layers
        "hidden_dim": 3072,              # Size of the intermediate dimension in FeedForward
        # head_dim 可以独立于 emb_dim/n_heads 设置(此处 emb_dim=1024, n_heads=16, emb_dim // n_heads = 64,
        # 但 head_dim 显式设为 128),说明 head_dim 并不要求等于 emb_dim // n_heads,GQA 层会用 head_dim 单独计算 Q/K/V 的输出维度
        "head_dim": 128,                 # Size of the heads in GQA
        # QK-Norm: 在计算注意力前对每个头的 Q、K 向量分别做 RMSNorm,是 Qwen3 相比 Qwen2 新增的稳定性技巧
        "qk_norm": True,                 # Whether to normalize queries and keys in GQA
        # GQA: n_heads=16 个 Query 头共享 n_kv_groups=8 组 K/V,即每 group_size = 16 // 8 = 2 个 Query 头共用一组 K/V,
        # 减少了 K/V(尤其是 KV cache)的显存占用,同时相比 MQA(只有1组)保留了更多的表达能力
        "n_kv_groups": 8,                # Key-Value groups for grouped-query attention
        "rope_base": 1_000_000.0,        # The base in RoPE's "theta"
        "dtype": torch.bfloat16,         # Lower-precision dtype to reduce memory usage
    }

elif CHOOSE_MODEL == "1.7B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 2048,                 # 2x larger than above
        "n_heads": 16,
        "n_layers": 28,
        "hidden_dim": 6144,              # 2x larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

elif CHOOSE_MODEL == "4B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 2560,                 # 25% larger than above
        "n_heads": 32,                   # 2x larger than above
        "n_layers": 36,                  # 29% larger than above
        "hidden_dim": 9728,              # ~3x larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

elif CHOOSE_MODEL == "8B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 4096,                 # 60% larger than above
        "n_heads": 32,
        "n_layers": 36,                  # 26% larger than above
        "hidden_dim": 12288,
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

elif CHOOSE_MODEL == "14B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 5120,                 # 25% larger than above
        "n_heads": 40,                   # 25% larger than above
        "n_layers": 40,                  # 11% larger than above
        "hidden_dim": 17408,             # 42% larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

elif CHOOSE_MODEL == "32B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 5120,
        "n_heads": 64,                   # 60% larger than above
        "n_layers": 64,                  # 60% larger than above
        "hidden_dim": 25600,             # 47% larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

else:
    raise ValueError(f"{CHOOSE_MODEL} is not supported.")
# 固定随机种子后实例化模型;由于随后会加载预训练权重覆盖所有参数,这里的随机初始化值本身并不重要
torch.manual_seed(123)
model = Qwen3Model(QWEN3_CONFIG)
model

In [ ]:
# 用一个简单的三词元输入做一次前向传播冒烟测试,验证模型结构可以正确运行(此时权重仍是随机初始化,输出无意义)
model(torch.tensor([1, 2, 3]).unsqueeze(0))

In [ ]:
# 统计模型的总参数量(若使用了权重绑定,tok_emb 的参数会被重复计入一次)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
# 如果模型使用了权重绑定(tok_emb 的权重矩阵与 out_head 复用同一份参数),那么上面统计的总参数量中 tok_emb 的部分
# 被重复计入了两次,因此这里减去一次 tok_emb 的参数量,得到真实的唯一参数数量
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
# 估算模型在给定 dtype 下的显存/内存占用(参数 + 梯度 + 缓冲区,例如 RoPE 的 cos/sin 表)
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())
    # 缓冲区包括 register_buffer 注册的 cos/sin 位置编码表,虽然不是可训练参数,但同样占用显存

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    # 这里简化处理: 假设推理时不需要梯度(纯推理场景下 total_grads 通常为 0),用给定 dtype 的单元素字节数估算整体占用
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
# 自动选择可用的计算设备,优先级: CUDA GPU > Apple Silicon 的 MPS > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 将模型的全部参数和缓冲区(包括 cos/sin 表)搬到目标设备上
model.to(device);

4. Load pretrained weights

In [ ]:
# 将从 HuggingFace 下载的 Qwen3 原始权重(键名形如 model.layers.{l}.xxx)按名称映射并拷贝进自定义模型的对应参数中
def load_weights_into_qwen(model, param_config, params):
    # 辅助函数: 先校验形状是否一致,再用 copy_ 原地写入,避免破坏原有的 dtype/device/nn.Parameter 等属性
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    for l in range(param_config["n_layers"]):
        block = model.trf_blocks[l]
        att = block.att

        # Q, K, V projections
        # 注意 W_query 权重形状是 (num_heads*head_dim, emb_dim),而 W_key/W_value 权重形状是 (num_kv_groups*head_dim, emb_dim),
        # 因为 GQA 下 K、V 的头数更少,这也是权重加载能直接对齐 HF checkpoint 里 GQA 形状的原因
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )

        # QK norms
        # 仅当模型配置了 qk_norm=True 时,HF 权重里才存在 q_norm/k_norm 的 scale 参数,故先用 hasattr/None 判断再加载
        if hasattr(att, "q_norm") and att.q_norm is not None:
            att.q_norm.scale = assign(
                att.q_norm.scale,
                params[f"model.layers.{l}.self_attn.q_norm.weight"],
                f"model.layers.{l}.self_attn.q_norm.weight"
            )
        if hasattr(att, "k_norm") and att.k_norm is not None:
            att.k_norm.scale = assign(
                att.k_norm.scale,
                params[f"model.layers.{l}.self_attn.k_norm.weight"],
                f"model.layers.{l}.self_attn.k_norm.weight"
            )

        # Attention layernorm
        block.norm1.scale = assign(
            block.norm1.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # Feedforward weights
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight"
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight"
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight"
        )
        block.norm2.scale = assign(
            block.norm2.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # Final normalization and output head
    model.final_norm.scale = assign(model.final_norm.scale, params["model.norm.weight"], "model.norm.weight")

    # 有些规模的 Qwen3(如 0.6B)输出层与词嵌入层权重绑定(weight tying),checkpoint 中不单独存 lm_head.weight,
    # 此时直接复用 tok_emb 的权重矩阵作为输出投影层的权重
    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")

In [ ]:
# 从 HuggingFace Hub 下载预训练权重: 小模型只有单个 safetensors 文件,大模型被切分为多个分片(shard)并附带索引文件
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download


if USE_REASONING_MODEL or USE_INSTRUCT_MODEL:
    repo_id = f"Qwen/Qwen3-{CHOOSE_MODEL}"
else:
    repo_id = f"Qwen/Qwen3-{CHOOSE_MODEL}-Base"

local_dir = Path(repo_id).parts[-1]

# 0.6B 模型体积较小,权重只有一个 safetensors 文件,直接下载并加载即可
if CHOOSE_MODEL == "0.6B":
    weights_file = hf_hub_download(
        repo_id=repo_id,
        filename="model.safetensors",
        local_dir=local_dir,
    )
    weights_dict = load_file(weights_file)
else:
    # 更大的模型权重被切分成多个分片文件,需要先下载整个仓库快照,再读取索引文件了解每个参数存放在哪个分片里
    repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
    index_path = os.path.join(repo_dir, "model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)

    # 依次加载每个分片文件并合并成一个完整的 state_dict
    weights_dict = {}
    for filename in set(index["weight_map"].values()):
        shard_path = os.path.join(repo_dir, filename)
        shard = load_file(shard_path)
        weights_dict.update(shard)

# 权重加载完成后,把随机初始化的参数替换为预训练权重,再次搬运到目标设备,并释放临时的 state_dict 以节省内存
load_weights_into_qwen(model, QWEN3_CONFIG, weights_dict)
model.to(device)
del weights_dict

3. Load tokenizer

In [ ]:
# 实现一个精简版 Qwen3 分词器: 基于 HuggingFace tokenizers 库的 BPE 编码,并手动处理特殊标记(special tokens)和聊天模板
import re
from tokenizers import Tokenizer

class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>"
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")
    # 用正则表达式把文本中的特殊标记(如 <|im_start|>、<think> 等)切分出来,切分后的特殊标记片段单独按 ID 映射,
    # 其余普通文本片段仍交给底层 BPE 分词器处理

    def __init__(self, tokenizer_file_path="tokenizer.json", repo_id=None,
                 apply_chat_template=True, add_generation_prompt=False, add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        # 注意: 这里用到的 Path 是在前面加载权重的 cell 中通过 `from pathlib import Path` 导入的,依赖 notebook 按顺序执行
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
    # 将文本编码为 token id 列表: 若整段文本本身就是一个特殊标记(如 "<|endoftext|>")则直接返回对应 id,
    # 否则按需先包装聊天模板,再用正则切出特殊标记片段单独映射,普通文本片段交给 BPE 分词器编码
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
    # 按 Qwen3 的对话模板拼接: <|im_start|>user\n{内容}<|im_end|>\n;如果需要生成回复,再补上助手开头标记,
    # 对推理模型可选择保留空的 <think></think> 块(不启用思考)或留空等待模型自行生成思考过程
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s
if USE_REASONING_MODEL:
    tokenizer_file_path = f"Qwen3-{CHOOSE_MODEL}/tokenizer.json"
else:
    tokenizer_file_path = f"Qwen3-{CHOOSE_MODEL}-Base/tokenizer.json"

hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)

if USE_REASONING_MODEL or USE_INSTRUCT_MODEL:
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_file_path,
        repo_id=repo_id,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=USE_REASONING_MODEL
    )

else:
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_file_path,
        repo_id=repo_id,
        apply_chat_template=False,
        add_generation_prompt=False,
        add_thinking=False
    )
# 用分词器把提示词编码为 token id,再解码回文本,仅作健全性检查(验证编码/解码可逆、聊天模板是否按预期拼接)
prompt = "Give me a short introduction to large language models."

input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

4. Generate text

In [ ]:
# 基于 KV cache 的流式自回归生成: 与不带缓存的实现相比,每一步只需要把新产生的 1 个 token 喂给模型,
# 而不必把已经生成的全部历史 token 重新过一遍网络,从而把每步解码的计算复杂度从 O(当前序列长度) 降到 O(1)
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None, context_size=None):
    model.eval()

    with torch.no_grad():
        cache = KVCache(n_layers=model.cfg["n_layers"])
        # 为每一层 Transformer 准备一个独立的 (key, value) 缓存槽位
        model.reset_kv_cache()

        # Prime the cache with the initial context
        # 首次前向传播把完整的提示词(prompt)一次性喂进去,为每一层生成初始的 K/V 缓存,
        # 同时得到最后一个位置的 logits 用于预测下一个 token
        logits = model(token_ids, cache=cache)

        for _ in range(max_new_tokens):
        # 之后每一步只需要处理新生成的单个 token
            next_token = torch.argmax(logits[:, -1], dim=-1, keepdim=True)

            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break

            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)
            # 这里拼接只是为了在函数外部维护/展示完整序列,并不会把拼接后的完整序列重新输入给模型(避免重复计算)

            # Feed only the new token to the model; cache handles history
            # 由于 KV cache 已保存之前所有位置的 key/value,这里只需把新 token 走一遍 QKV 投影和 RoPE
            # (用当前累计位置 current_pos 作为 offset),再与缓存中的历史 K/V 拼接做注意力计算,无需重算历史 token 的 K/V
            logits = model(next_token, cache=cache)
# 下面是对上面流式生成函数的实际调用,并统计生成速度(tokens/sec)与显存占用
import time

input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
generated_tokens = 0

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    generated_tokens += 1
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

elapsed = time.perf_counter() - start_time
tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")